<a href="https://colab.research.google.com/github/yatinbansal/rag-application/blob/main/RAG_Capstone_v2_SecureStorage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 RAG Pipeline — RAGBench Capstone (v2 — Secure Storage)
---
## 🔐 Storage Strategy — Choose One

| | **Option 1** | **Option 2** |
|---|---|---|
| **Mechanism** | Google Drive mount (scoped to one folder) | Manual upload/download, no Drive access |
| **Security** | Drive mounted; all code stays in `RAG_Capstone/` only | ✅ No Drive access at all |
| **Convenience** | ✅ Persists across sessions automatically | 🔄 Manual upload each new session |
| **Best for** | Personal accounts with no sensitive Drive content | Extra caution / shared notebooks |

➡️ **Set `STORAGE_OPTION = 1` or `STORAGE_OPTION = 2` in Section 0.**

---

## 📋 Table of Contents

| # | Section | Expensive? |
|---|---------|------------|
| 0 | **Config** | — |
| 1 | **Environment Setup** | Once/session |
| 2 | **Storage Setup** | Once/session |
| 3 | **Dataset Loading** | Fast |
| 4 | **Document Chunking** ✅ Checkpointed | ~1 min |
| 5 | **Embedding + FAISS** ✅ Checkpointed | ~5 min |
| 6 | **Retrieval Functions** | Fast |
| 7 | **LLM Generation** | Fast |
| 8 | **Judge LLM + TRACe Metrics** | Fast |
| 9 | **Run Full Evaluation** ✅ Checkpointed | ~10 min |
| 10 | **RMSE & AUC-ROC** | Fast |
| 11 | **Multi-Dataset Comparison** | Fast |
| 12 | **Gradio Demo** | Run last |


---
# 📌 Section 0 — Configuration
> **Only change this cell between experiments.**


In [ ]:
# ============================================================
# SECTION 0: CONFIGURATION
# Change ONLY this cell between experiments.
# ============================================================

# Dataset options: "finqa","hotpotqa","emanual","techqa","covidqa","pubmedqa","tatqa"
DATASET_NAME     = "finqa"

# Storage: 1 = Google Drive (scoped)  |  2 = Manual upload/download
STORAGE_OPTION   = 2

# Embedding model
# "all-MiniLM-L6-v2"                         fast baseline (384-dim)
# "BAAI/bge-small-en-v1.5"                   better (384-dim)
# "BAAI/bge-base-en-v1.5"                    best (768-dim)
# "finlang/finance-embeddings-investopedia"   domain-specific (finqa)
# "nlpaueb/legal-bert-base-uncased"           domain-specific (legal)
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
USE_RERANKER     = True

# LLM options: "llama-3.1-8b-instant", "gemma2-9b-it", "llama-3.3-70b-versatile"
LLM_MODEL        = "llama-3.1-8b-instant"
JUDGE_MODEL      = "llama-3.3-70b-versatile"

CHUNK_SIZE       = 512
CHUNK_OVERLAP    = 50
TOP_K_INITIAL    = 15   # FAISS candidates before reranking
TOP_K_FINAL      = 5    # Final chunks sent to LLM
NUM_EVAL_SAMPLES = 50

# ── Derived paths (do not edit) ─────────────────────────────
import os
EMBED_SAFE          = EMBED_MODEL_NAME.replace("/", "_")
LOCAL_CACHE         = f"/content/rag_cache/{DATASET_NAME}"
DRIVE_PROJECT_FOLDER= "RAG_Capstone"
DRIVE_BASE          = f"/content/drive/MyDrive/{DRIVE_PROJECT_FOLDER}/{DATASET_NAME}"

CHUNKS_FILE  = f"{LOCAL_CACHE}/chunks_{CHUNK_SIZE}_{CHUNK_OVERLAP}.pkl"
FAISS_FILE   = f"{LOCAL_CACHE}/faiss_{EMBED_SAFE}.index"
EMBEDS_FILE  = f"{LOCAL_CACHE}/embeddings_{EMBED_SAFE}.npy"
RESULTS_FILE = f"{LOCAL_CACHE}/results_{EMBED_SAFE}_top{TOP_K_FINAL}_n{NUM_EVAL_SAMPLES}.csv"

CHUNKS_FNAME  = f"chunks_{DATASET_NAME}_{CHUNK_SIZE}_{CHUNK_OVERLAP}.pkl"
FAISS_FNAME   = f"faiss_{DATASET_NAME}_{EMBED_SAFE}.index"
EMBEDS_FNAME  = f"embeddings_{DATASET_NAME}_{EMBED_SAFE}.npy"
RESULTS_FNAME = f"results_{DATASET_NAME}_{EMBED_SAFE}_top{TOP_K_FINAL}_n{NUM_EVAL_SAMPLES}.csv"

os.makedirs(LOCAL_CACHE, exist_ok=True)

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"  Dataset:      {DATASET_NAME}")
print(f"  Storage:      Option {STORAGE_OPTION} ({"Google Drive (scoped)" if STORAGE_OPTION==1 else "Manual upload/download"})")
print(f"  Embed:        {EMBED_MODEL_NAME}")
print(f"  Reranker:     {RERANKER_MODEL_NAME} ({"ON" if USE_RERANKER else "OFF"})")
print(f"  LLM:          {LLM_MODEL}")
print(f"  Chunk:        {CHUNK_SIZE}, overlap {CHUNK_OVERLAP}")
print(f"  Top-K:        {TOP_K_INITIAL} -> {TOP_K_FINAL}")
print(f"  Eval samples: {NUM_EVAL_SAMPLES}")
if STORAGE_OPTION == 1:
    print(f"  Drive folder: MyDrive/{DRIVE_PROJECT_FOLDER}/{DATASET_NAME}/")
    print("  NOTE: Only this folder is ever written to.")
else:
    print(f"  Local cache:  {LOCAL_CACHE}")
    print("  NOTE: Google Drive is NOT mounted.")
print("=" * 60)


CONFIGURATION
  Dataset:      finqa
  Storage:      Option 2 (Manual upload/download)
  Embed:        all-MiniLM-L6-v2
  Reranker:     cross-encoder/ms-marco-MiniLM-L-6-v2 (ON)
  LLM:          llama-3.1-8b-instant
  Chunk:        512, overlap 50
  Top-K:        15 -> 5
  Eval samples: 50
  Local cache:  /content/rag_cache/finqa
  NOTE: Google Drive is NOT mounted.


---
# 📌 Section 1 — Environment Setup
> Install all libraries and imports. Run once per session.


In [ ]:
# SECTION 1A: Install Libraries (all in one cell)
!pip install -q datasets sentence-transformers faiss-cpu \
  langchain langchain-community langchain-text-splitters \
  groq gradio scikit-learn huggingface_hub rank_bm25
print("All libraries installed!")


All libraries installed!


In [ ]:
# SECTION 1B: All Imports
import os, json, time, pickle, re, shutil
import numpy as np
import pandas as pd
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.metrics import mean_squared_error, roc_auc_score
from groq import Groq
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    print("GROQ_API_KEY not found - add it via left panel -> Secrets")

groq_client = Groq()
print("All imports ready.")


GROQ_API_KEY loaded from Colab Secrets
All imports ready.


---
# 📌 Section 2 — Storage Setup
> Runs the storage strategy you chose in Section 0.
> - **Option 1**: Mounts Google Drive, scoped ONLY to `MyDrive/RAG_Capstone/`. All operations stay inside that folder.
> - **Option 2**: No Drive access. Helpers download files to your machine and let you re-upload next session.


In [ ]:
# ============================================================
# SECTION 2: Storage Setup
# Behaviour is controlled entirely by STORAGE_OPTION in Section 0.
# ============================================================

if STORAGE_OPTION == 1:
    # ----------------------------------------------------------
    # OPTION 1: Google Drive (Scoped)
    # ----------------------------------------------------------
    print("OPTION 1 - Google Drive (Scoped Mount)")
    print("-" * 52)
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_BASE, exist_ok=True)

    print(f"Drive mounted.")
    print(f"Project folder: MyDrive/{DRIVE_PROJECT_FOLDER}/{DATASET_NAME}/")
    print()
    print("Security notes:")
    print("  All read/write is restricted to:", DRIVE_BASE)
    print("  Do NOT store Atlassian credentials or work files in Drive.")
    print("  Do NOT share a runtime where Drive is mounted.")

    # Helper: pickle checkpoint
    def save_checkpoint(obj, local_path):
        with open(local_path, "wb") as f:
            pickle.dump(obj, f)
        drive_path = local_path.replace(LOCAL_CACHE, DRIVE_BASE)
        os.makedirs(os.path.dirname(drive_path), exist_ok=True)
        shutil.copy2(local_path, drive_path)
        print(f"  Saved + mirrored to Drive: {os.path.basename(drive_path)}")

    def load_checkpoint(local_path):
        if os.path.exists(local_path):
            with open(local_path, "rb") as f:
                return pickle.load(f)
        drive_path = local_path.replace(LOCAL_CACHE, DRIVE_BASE)
        if os.path.exists(drive_path):
            shutil.copy2(drive_path, local_path)
            print(f"  Pulled from Drive: {os.path.basename(drive_path)}")
            with open(local_path, "rb") as f:
                return pickle.load(f)
        return None

    def checkpoint_exists(local_path):
        drive_path = local_path.replace(LOCAL_CACHE, DRIVE_BASE)
        return os.path.exists(local_path) or os.path.exists(drive_path)

    def save_faiss(index, embs, ipath, epath):
        faiss.write_index(index, ipath)
        np.save(epath, embs)
        for lp in [ipath, epath]:
            dp = lp.replace(LOCAL_CACHE, DRIVE_BASE)
            shutil.copy2(lp, dp)
        print("  FAISS index + embeddings saved to Drive.")

    def load_faiss(ipath, epath):
        if not os.path.exists(ipath):
            di = ipath.replace(LOCAL_CACHE, DRIVE_BASE)
            de = epath.replace(LOCAL_CACHE, DRIVE_BASE)
            if os.path.exists(di):
                shutil.copy2(di, ipath)
                shutil.copy2(de, epath)
                print("  FAISS index pulled from Drive.")
        if os.path.exists(ipath):
            return faiss.read_index(ipath), np.load(epath)
        return None, None

    def save_results(df, csv_path):
        df.to_csv(csv_path, index=False)
        dp = csv_path.replace(LOCAL_CACHE, DRIVE_BASE)
        df.to_csv(dp, index=False)
        print(f"  Results saved to Drive: {os.path.basename(dp)}")

    def load_results(csv_path):
        if os.path.exists(csv_path):
            return pd.read_csv(csv_path)
        dp = csv_path.replace(LOCAL_CACHE, DRIVE_BASE)
        if os.path.exists(dp):
            return pd.read_csv(dp)
        return None

    def prompt_upload_checkpoint(local_path): return False
    def prompt_upload_faiss(ipath, epath): return False
    def prompt_upload_results(csv_path): return None

    print("Option 1 storage helpers ready.")

else:
    # ----------------------------------------------------------
    # OPTION 2: Manual Upload / Download (No Drive Access)
    # ----------------------------------------------------------
    print("OPTION 2 - Manual Upload/Download (No Drive mounted)")
    print("-" * 52)
    print("Google Drive is NOT accessed.")
    print("Workflow:")
    print("  1. Checkpoints saved locally in this session.")
    print("  2. Files auto-downloaded to your machine after creation.")
    print("  3. Upload them back at the start of the next session.")
    from google.colab import files as colab_files

    def save_checkpoint(obj, local_path):
        with open(local_path, "wb") as f:
            pickle.dump(obj, f)
        print(f"  Saved: {os.path.basename(local_path)}")
        print("  Downloading to your machine...")
        colab_files.download(local_path)

    def load_checkpoint(local_path):
        if os.path.exists(local_path):
            with open(local_path, "rb") as f:
                return pickle.load(f)
        return None

    def checkpoint_exists(local_path):
        return os.path.exists(local_path)

    def prompt_upload_checkpoint(local_path):
        fname = os.path.basename(local_path)
        print(f"  Upload '{fname}' from a previous session (or skip to rebuild):")
        uploaded = colab_files.upload()
        if fname in uploaded:
            with open(local_path, "wb") as f:
                f.write(uploaded[fname])
            print(f"  Loaded '{fname}'")
            return True
        print(f"  '{fname}' not uploaded. Will rebuild.")
        return False

    def save_faiss(index, embs, ipath, epath):
        faiss.write_index(index, ipath)
        np.save(epath, embs)
        print("  Saved FAISS index + embeddings locally.")
        print("  Downloading both to your machine...")
        colab_files.download(ipath)
        colab_files.download(epath)

    def load_faiss(ipath, epath):
        if os.path.exists(ipath) and os.path.exists(epath):
            return faiss.read_index(ipath), np.load(epath)
        return None, None

    def prompt_upload_faiss(ipath, epath):
        for path in [ipath, epath]:
            fname = os.path.basename(path)
            print(f"  Upload '{fname}':")
            uploaded = colab_files.upload()
            if fname in uploaded:
                with open(path, "wb") as f:
                    f.write(uploaded[fname])
            else:
                print(f"  '{fname}' not found - will rebuild.")
                return False
        return True

    def save_results(df, csv_path):
        df.to_csv(csv_path, index=False)
        print("  Results CSV saved locally.")
        print("  Downloading...")
        colab_files.download(csv_path)

    def load_results(csv_path):
        if os.path.exists(csv_path):
            return pd.read_csv(csv_path)
        return None

    def prompt_upload_results(csv_path):
        fname = os.path.basename(csv_path)
        print(f"  Upload '{fname}' if available:")
        uploaded = colab_files.upload()
        if fname in uploaded:
            with open(csv_path, "wb") as f:
                f.write(uploaded[fname])
            return pd.read_csv(csv_path)
        return None

    print("Option 2 storage helpers ready.")

print()
print("Storage initialised. Proceed to Section 3.")


OPTION 2 - Manual Upload/Download (No Drive mounted)
----------------------------------------------------
Google Drive is NOT accessed.
Workflow:
  1. Checkpoints saved locally in this session.
  2. Files auto-downloaded to your machine after creation.
  3. Upload them back at the start of the next session.
Option 2 storage helpers ready.

Storage initialised. Proceed to Section 3.


---
# 📌 Section 3 — Dataset Loading & Exploration
> Fast — no checkpoint needed.


In [ ]:
# SECTION 3A: HF Auth + Load Dataset
from huggingface_hub import login
try:
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("HF authenticated via Colab Secrets")
except Exception:
    print("HF_TOKEN not in Secrets - trying interactive...")
    login()

print(f"Loading RAGBench: {DATASET_NAME} (test split)...")
ds = load_dataset("rungalileo/ragbench", DATASET_NAME, split="test")
print(f"Loaded {len(ds)} samples | Columns: {len(ds.column_names)}")


HF authenticated via Colab Secrets
Loading RAGBench: finqa (test split)...
Loaded 2294 samples | Columns: 26


In [ ]:
# SECTION 3B: Explore Dataset
s = ds[0]
print("SAMPLE ENTRY")
print(f"  Question:    {s['question'][:100]}...")
print(f"  # Docs:      {len(s['documents'])}")
print(f"  Response:    {s['response'][:100]}...")
print(f"  Adherence:   {s['adherence_score']}")
print(f"  Relevance:   {s['relevance_score']:.4f}")
print(f"  Utilization: {s['utilization_score']:.4f}")
print(f"  Completeness:{s['completeness_score']:.4f}")

all_flat, doc_set = [], set()
for sample in ds:
    for doc in sample["documents"]:
        all_flat.append(doc)
        doc_set.add(doc)
lengths = [len(d) for d in doc_set]
print(f"\nDOCUMENT STATS")
print(f"  Unique docs:  {len(doc_set)}")
print(f"  Dedup ratio:  {len(doc_set)/len(all_flat):.1%}")
print(f"  Avg length:   {np.mean(lengths):.0f} chars")
print(f"  Max length:   {max(lengths)} chars")

print(f"\nGROUND-TRUTH METRIC DISTRIBUTIONS")
for col in ["relevance_score","utilization_score","completeness_score","adherence_score"]:
    vals = [x[col] for x in ds if x[col] is not None]
    if isinstance(vals[0], bool): vals = [1.0 if v else 0.0 for v in vals]
    print(f"  {col:28s}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}")


SAMPLE ENTRY
  Question:    what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?...
  # Docs:      3
  Response:    The rate of return in Cadence Design Systems Inc. from 2010 to 2011 is 37.9%. This is calculated by ...
  Adherence:   True
  Relevance:   0.1111
  Utilization: 0.1111
  Completeness:1.0000

DOCUMENT STATS
  Unique docs:  1097
  Dedup ratio:  16.4%
  Avg length:   1342 chars
  Max length:   6693 chars

GROUND-TRUTH METRIC DISTRIBUTIONS
  relevance_score               mean=0.080  std=0.081
  utilization_score             mean=0.068  std=0.067
  completeness_score            mean=0.914  std=0.217
  adherence_score               mean=0.915  std=0.280


---
# 📌 Section 4 — Document Chunking  ✅ *Checkpointed*
> **Option 1**: Auto-loaded from Drive / saved to Drive.
> **Option 2**: Auto-downloaded after creation; upload prompt at start of new session.


In [ ]:
# SECTION 4: Document Chunking (checkpoint-guarded)

loaded = load_checkpoint(CHUNKS_FILE)

if loaded is None and STORAGE_OPTION == 2:
    print(f"No chunks in session. Upload '{CHUNKS_FNAME}' if available (or skip).")
    if prompt_upload_checkpoint(CHUNKS_FILE):
        loaded = load_checkpoint(CHUNKS_FILE)

if loaded is not None:
    chunks, chunk_to_doc, all_documents = loaded
    print(f"Chunks loaded: {len(chunks)} chunks from {len(all_documents)} docs")
else:
    print(f"Building chunks for '{DATASET_NAME}'...")
    all_documents, seen = [], set()
    for sample in ds:
        for doc in sample["documents"]:
            if doc not in seen:
                seen.add(doc)
                all_documents.append(doc)
    print(f"  Unique docs: {len(all_documents)}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        separators=["\n\n", "\n", ". ", ", ", " ", ""]
    )
    chunks, chunk_to_doc = [], []
    for idx, doc in enumerate(all_documents):
        for chunk in splitter.split_text(doc):
            chunks.append(chunk)
            chunk_to_doc.append(idx)

    print(f"  Created {len(chunks)} chunks | avg {np.mean([len(c) for c in chunks]):.0f} chars")
    save_checkpoint((chunks, chunk_to_doc, all_documents), CHUNKS_FILE)

# Distribution
print("\nCHUNK LENGTH DISTRIBUTION")
for lo, hi in [(0,100),(100,200),(200,300),(300,400),(400,CHUNK_SIZE),(CHUNK_SIZE,99999)]:
    cnt = sum(1 for c in chunks if lo <= len(c) < hi)
    bar = chr(9608) * max(0, cnt * 30 // len(chunks))
    hi_label = str(hi) if hi < 99999 else "inf"
    print(f"  {lo:>5}-{hi_label:>5} chars: {cnt:>5}  {bar}")


Chunks loaded: 3978 chunks from 1097 docs

CHUNK LENGTH DISTRIBUTION
      0-  100 chars:   109  
    100-  200 chars:   311  ██
    200-  300 chars:   522  ███
    300-  400 chars:  1021  ███████
    400-  512 chars:  1999  ███████████████
    512-  inf chars:    16  


---
# 📌 Section 5 — Embedding + FAISS Index  ✅ *Checkpointed*
> Most expensive step (~5 min for finqa). Saved automatically per storage option.


In [ ]:
# SECTION 5A: Load Embedding Model
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
embed_dim = embed_model.get_embedding_dimension()
print(f"  Loaded! Dimension: {embed_dim}")


Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Loaded! Dimension: 384


In [ ]:
# SECTION 5B: Build or Load FAISS Index (checkpoint-guarded)

index, chunk_embeddings = load_faiss(FAISS_FILE, EMBEDS_FILE)

if index is None and STORAGE_OPTION == 2:
    print(f"No FAISS index in session.")
    print(f"Upload '{FAISS_FNAME}' and '{EMBEDS_FNAME}' if available (or skip to rebuild).")
    if prompt_upload_faiss(FAISS_FILE, EMBEDS_FILE):
        index, chunk_embeddings = load_faiss(FAISS_FILE, EMBEDS_FILE)

if index is not None:
    print(f"FAISS index loaded: {index.ntotal} vectors x {embed_dim}d")
else:
    print(f"Building index: embedding {len(chunks)} chunks (3-10 min)...")
    start = time.time()
    chunk_embeddings = embed_model.encode(
        chunks, show_progress_bar=True, batch_size=64, normalize_embeddings=True
    )
    print(f"  Done in {time.time()-start:.1f}s | Shape: {chunk_embeddings.shape}")

    index = faiss.IndexFlatIP(embed_dim)
    index.add(chunk_embeddings.astype("float32"))
    print(f"  {index.ntotal} vectors indexed")

    save_faiss(index, chunk_embeddings, FAISS_FILE, EMBEDS_FILE)

print(f"FAISS ready: {index.ntotal} vectors x {embed_dim}d")


FAISS index loaded: 3978 vectors x 384d
FAISS ready: 3978 vectors x 384d


---
# 📌 Section 6 — Retrieval Functions


In [ ]:
# SECTION 6A: Basic FAISS Retrieval
def retrieve(query, top_k=TOP_K_FINAL):
    """Stage 1: Embed query -> FAISS search -> return top_k chunks."""
    qemb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(qemb, top_k)
    return [chunks[i] for i in idxs[0]], scores[0].tolist(), idxs[0].tolist()

print("retrieve(query, top_k) defined")


retrieve(query, top_k) defined


In [ ]:
# SECTION 6B: Two-Stage Retrieval with Reranker

if USE_RERANKER:
    print(f"Loading reranker: {RERANKER_MODEL_NAME}")
    reranker = CrossEncoder(RERANKER_MODEL_NAME)
    print("  Reranker loaded!")
else:
    reranker = None
    print("Reranker disabled (USE_RERANKER = False)")


def retrieve_and_rerank(query, initial_top_k=TOP_K_INITIAL, final_top_k=TOP_K_FINAL):
    """Stage 1: FAISS candidates. Stage 2: CrossEncoder reranks."""
    if reranker is None:
        return retrieve(query, top_k=final_top_k)
    qemb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(qemb, initial_top_k)
    candidates = [(chunks[i], scores[0][r], i) for r, i in enumerate(idxs[0])]
    rscores = reranker.predict([[query, c[0]] for c in candidates])
    reranked = sorted(zip(candidates, rscores), key=lambda x: x[1], reverse=True)
    return ([r[0][0] for r in reranked[:final_top_k]],
            [float(r[1]) for r in reranked[:final_top_k]],
            [r[0][2] for r in reranked[:final_top_k]])

print("retrieve_and_rerank(query) defined")


Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  Reranker loaded!
retrieve_and_rerank(query) defined


In [ ]:
# SECTION 6C: Quick Test
test_q  = ds[0]["question"]
gt_docs = ds[0]["documents"]
print(f"Query: {test_q[:70]}...\n")

r_chunks, r_scores, _ = retrieve(test_q, top_k=5)
print("--- FAISS Only ---")
for rank, (chunk, score) in enumerate(zip(r_chunks, r_scores), 1):
    hit = any(chunk[:80] in gt for gt in gt_docs)
    print(f"  {rank}. {score:.4f} | {"HIT" if hit else "---"} | {chunk[:70]}...")

if USE_RERANKER:
    rr_chunks, rr_scores, _ = retrieve_and_rerank(test_q)
    print("\n--- FAISS + Reranker ---")
    for rank, (chunk, score) in enumerate(zip(rr_chunks, rr_scores), 1):
        hit = any(chunk[:80] in gt for gt in gt_docs)
        print(f"  {rank}. {score:.4f} | {"HIT" if hit else "---"} | {chunk[:70]}...")


Query: what is the rate of return in cadence design systems inc . of an inves...

--- FAISS Only ---
  1. 0.6069 | HIT | . the graph assumes that the value of the investment in our common sto...
  2. 0.5927 | --- | is based on an asset allocation assumption of 25% ( 25 % ) global equi...
  3. 0.5369 | --- | . in determining the long-term rate of return for a plan , we consider...
  4. 0.5179 | --- | . the comparison assumes $ 100 was invested on october 27 , 2013 in ap...
  5. 0.5176 | --- | . the comparison assumes $ 100 was invested on october 26 , 2008 in ap...

--- FAISS + Reranker ---
  1. 5.0747 | HIT | . the graph assumes that the value of the investment in our common sto...
  2. -1.6854 | HIT | . , the nasdaq composite index , and s&p 400 information technology ca...
  3. -2.4903 | --- | 2013 . in 2011 , asset returns were lower than expected by $ 471 milli...
  4. -2.7710 | --- | is based on an asset allocation assumption of 25% ( 25 % ) global equi...
  5. -3.2250 | --- | . w

---
# 📌 Section 7 — LLM Generation (Groq)


In [ ]:
# SECTION 7A: Test LLM
resp = groq_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role":"user","content":"Say RAG pipeline ready! in one line."}],
    temperature=0.0, max_tokens=30
)
print(f"LLM says: {resp.choices[0].message.content}")
print(f"Model: {resp.model} | Tokens: {resp.usage.total_tokens}")


LLM says: RAG pipeline ready!
Model: llama-3.1-8b-instant | Tokens: 51


In [ ]:
# SECTION 7B: generate_answer() function
def generate_answer(question, context_docs, model=LLM_MODEL):
    """Generate answer from retrieved context. Context-only, no hallucination."""
    context = "\n\n---\n\n".join(
        [f"[Document {i+1}]:\n{doc}" for i, doc in enumerate(context_docs)]
    )
    prompt = (
        "You are a helpful assistant that answers ONLY from the provided context.\n"
        "Rules:\n"
        "1. Use ONLY information from the context below.\n"
        "2. If context is insufficient say: I cannot answer this question based on the provided context.\n"
        "3. Be concise and precise. Quote specific numbers or facts.\n"
        "4. No outside knowledge.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\nAnswer:"
    )
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}],
        temperature=0.1, max_tokens=512
    )
    return response.choices[0].message.content

print("generate_answer(question, context_docs) defined")


generate_answer(question, context_docs) defined


In [ ]:
# SECTION 7C: End-to-End Test (1 sample)
s0 = ds[1]
q0 = s0["question"]
ctx, scores, _ = retrieve_and_rerank(q0) if USE_RERANKER else retrieve(q0)
print(f"Question: {q0}\n")
print(f"Top retrieval score: {scores[0]:.3f}\n")
answer = generate_answer(q0, ctx)
print(f"RAG Answer:\n{answer}\n")
print(f"Ground Truth:\n{s0['response'][:400]}")


Question: what is the ratio of the total american personnel to us airways personnel

Top retrieval score: 1.040

RAG Answer:
To find the ratio, we need to calculate the total personnel for American and US Airways. 

From Document 2, the total personnel for American is 61600 and for US Airways is 32800.

The ratio of American personnel to US Airways personnel is 61600 : 32800. 

To simplify, we can divide both numbers by 32800. 

The ratio is approximately 1.88 : 1.

Ground Truth:
The total number of American personnel is 61,600 and the total number of US Airways personnel is 32,800. 

Therefore, the ratio of total American personnel to US Airways personnel is 61,600:32,800 which simplifies to 308:164, which further simplifies to 77:41.


---
# 📌 Section 8 — Judge LLM + TRACe Metrics


In [ ]:
# SECTION 8A: build_context_with_keys()
def build_context_with_keys(context_docs):
    """Split docs into sentences, assign keys like 0a, 0b, 1a..."""
    ctx_str, all_keys = "", []
    for di, doc in enumerate(context_docs):
        sents = [s.strip() for s in doc.split(". ") if s.strip()]
        for si, sent in enumerate(sents):
            key = (f"{di}{chr(97+si)}" if si < 26
                   else f"{di}{chr(97+si//26-1)}{chr(97+si%26)}")
            ctx_str += f"[{key}]: {sent}.\n"
            all_keys.append(key)
    return ctx_str, all_keys

print("build_context_with_keys() defined")


build_context_with_keys() defined


In [ ]:
# ============================================================
# SECTION 8B: judge_response() — Strict Judge with Retries
# ============================================================

import json
import re
import time

def _safe_judge_output(total, all_keys):
    """Safe fallback when judge fails after all retries."""
    return {
        'all_relevant_sentence_keys': [],
        'all_utilized_sentence_keys': [],
        'sentence_support_information': [],
        'overall_supported': False,
        'total_context_sentences': total,
        'all_context_keys': all_keys,
        'judge_failed': True   # Flag so you can filter these out of RMSE later
    }


def _extract_json(raw_text):
    """
    Extract valid JSON from LLM output, handling:
    - Markdown code fences (```json ... ```)
    - Leading/trailing text around the JSON
    - Nested braces
    """
    # Strip markdown code fences
    cleaned = raw_text.strip()
    cleaned = re.sub(r'^```json\s*', '', cleaned)
    cleaned = re.sub(r'^```\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)
    cleaned = cleaned.strip()

    # Try direct parse first
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Find the outermost { ... } block
    brace_start = cleaned.find('{')
    if brace_start == -1:
        raise json.JSONDecodeError("No JSON object found", cleaned, 0)

    depth = 0
    for i in range(brace_start, len(cleaned)):
        if cleaned[i] == '{':
            depth += 1
        elif cleaned[i] == '}':
            depth -= 1
            if depth == 0:
                return json.loads(cleaned[brace_start:i+1])

    # Last resort: try from the brace to the end
    return json.loads(cleaned[brace_start:])


def _validate_judge_output(parsed):
    """
    Validate and normalize the parsed judge JSON.
    Ensures all required keys exist with correct types.
    """
    # Required list fields — default to empty list if missing or wrong type
    for key in ['all_relevant_sentence_keys', 'all_utilized_sentence_keys']:
        if key not in parsed or not isinstance(parsed[key], list):
            parsed[key] = []
        # Ensure all values are strings
        parsed[key] = [str(k) for k in parsed[key]]

    # sentence_support_information — must be a list of dicts
    ssi = parsed.get('sentence_support_information', [])
    if not isinstance(ssi, list):
        ssi = []
    validated_ssi = []
    for item in ssi:
        if isinstance(item, dict):
            validated_ssi.append({
                'response_sentence': str(item.get('response_sentence', '')),
                'supporting_keys': [str(k) for k in item.get('supporting_keys', [])],
                'supported': bool(item.get('supported', False))
            })
    parsed['sentence_support_information'] = validated_ssi

    # overall_supported — must be bool
    parsed['overall_supported'] = bool(parsed.get('overall_supported', False))

    return parsed


def judge_response(question, context_docs, response, model=JUDGE_MODEL, max_retries=2):
    """
    Use a Judge LLM to evaluate the RAG output.

    Extracts:
    - all_relevant_sentence_keys: context sentences relevant to the question
    - all_utilized_sentence_keys: context sentences used in the response
    - sentence_support_information: per-response-sentence support check
    - overall_supported: bool

    Features:
    - Retry logic (up to max_retries) on JSON parse failures
    - Robust JSON extraction (handles markdown fences, extra text)
    - Output validation (ensures correct types)
    - Exponential backoff on rate limit errors
    - judge_failed flag on final failure (so you can exclude from RMSE)

    Args:
        question:     The user's question
        context_docs: List of retrieved chunk texts
        response:     The generated RAG answer
        model:        Judge model name (use 70B for best results)
        max_retries:  Number of retries on failure

    Returns:
        dict with evaluation attributes
    """
    ctx_str, all_keys = build_context_with_keys(context_docs)
    total = len(all_keys)

    # Guard: if context or response is empty, return safe defaults
    if total == 0 or not response or not response.strip():
        print("   ⚠️  Empty context or response — skipping judge.")
        return _safe_judge_output(total, all_keys)

    # --- Build the judge prompt ---
    # Finance-aware: explicitly tells judge how to handle tables and numbers
    prompt = (
        "You are a STRICT evaluation judge for a RAG (Retrieval-Augmented Generation) system.\n\n"

        "DEFINITIONS — apply these precisely:\n"
        "- RELEVANT: A context sentence is relevant ONLY if it DIRECTLY contains a fact, "
        "number, or piece of information needed to answer the question. "
        "Sentences that are tangentially related, provide background, or mention "
        "similar but different entities are NOT relevant.\n"
        "- UTILIZED: A context sentence is utilized ONLY if specific information "
        "from that sentence (a number, name, date, or fact) actually appears in or "
        "directly informs the response.\n"
        "- SUPPORTED: A response sentence is supported ONLY if EVERY factual claim "
        "in it can be directly verified from the context sentences. "
        "Calculations are supported if the source numbers are in the context. "
        "Opinions, hedging, or qualifications not in context are NOT supported.\n\n"

        "SPECIAL RULES FOR FINANCIAL DATA:\n"
        "- Table data (JSON arrays, CSV-style rows) should be treated as individual "
        "data points. A sentence containing a table is relevant if the table has "
        "the numbers needed to answer.\n"
        "- If the response performs a calculation (percentage, ratio, difference), "
        "it is SUPPORTED if the input numbers come from the context, even if the "
        "context doesn't state the final calculated result.\n\n"

        f"Context (with sentence keys):\n{ctx_str}\n\n"
        f"Question: {question}\n\n"
        f"Response: {response}\n\n"

        "Return ONLY valid JSON with this EXACT structure (no text before or after):\n"
        "{\n"
        '  "all_relevant_sentence_keys": ["0a", "1b"],\n'
        '  "all_utilized_sentence_keys": ["0a"],\n'
        '  "sentence_support_information": [\n'
        '    {"response_sentence": "The return was 37.9%.", '
        '"supporting_keys": ["0a", "1b"], "supported": true}\n'
        "  ],\n"
        '  "overall_supported": true\n'
        "}"
    )

    # --- Retry loop ---
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            result = groq_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=2048
            )

            raw = result.choices[0].message.content

            # Extract and parse JSON
            parsed = _extract_json(raw)

            # Validate structure
            parsed = _validate_judge_output(parsed)

            # Sanity check: at least one field should be non-empty
            # (unless the response is truly "I cannot answer")
            is_refusal = ("cannot answer" in response.lower()
                          or "insufficient information" in response.lower())

            if (not is_refusal
                and len(parsed['all_relevant_sentence_keys']) == 0
                and len(parsed['sentence_support_information']) == 0
                and attempt < max_retries):
                print(f"   ⚠️  Judge returned all-empty on attempt {attempt+1} — retrying...")
                time.sleep(2)
                continue

            # Add metadata
            parsed['total_context_sentences'] = total
            parsed['all_context_keys'] = all_keys
            parsed['judge_failed'] = False
            return parsed

        except json.JSONDecodeError as e:
            last_error = f"JSON parse: {e}"
            if attempt < max_retries:
                print(f"   ⚠️  JSON parse failed (attempt {attempt+1}/{max_retries+1}) — retrying...")
                time.sleep(2 * (attempt + 1))   # Exponential backoff
            continue

        except Exception as e:
            error_str = str(e)
            last_error = error_str

            # Handle rate limits with smart backoff
            if "rate_limit" in error_str.lower() or "429" in error_str:
                wait_time = 5 * (attempt + 1)
                print(f"   ⚠️  Rate limited — waiting {wait_time}s (attempt {attempt+1})...")
                time.sleep(wait_time)
                continue

            if attempt < max_retries:
                print(f"   ⚠️  Judge error (attempt {attempt+1}): {error_str[:100]} — retrying...")
                time.sleep(3)
            continue

    # All retries exhausted
    print(f"   ❌ Judge failed after {max_retries+1} attempts. Last error: {last_error}")
    return _safe_judge_output(total, all_keys)


print("✅ judge_response() defined")
print(f"   Model: {JUDGE_MODEL}")
print(f"   Retries: 2 (with exponential backoff)")
print(f"   Features: JSON extraction, output validation, rate-limit handling")

✅ judge_response() defined
   Model: llama-3.3-70b-versatile
   Retries: 2 (with exponential backoff)
   Features: JSON extraction, output validation, rate-limit handling


In [ ]:
# SECTION 8C: compute_metrics() - TRACe Formulas
def compute_metrics(j):
    """TRACe formulas (RAGBench paper)."""
    rel   = set(j.get("all_relevant_sentence_keys", []))
    util  = set(j.get("all_utilized_sentence_keys", []))
    total = max(j.get("total_context_sentences", 1), 1)
    ctx_rel  = len(rel) / total
    ctx_util = len(util) / total
    complt   = len(rel & util) / len(rel) if rel else 0.0
    si = j.get("sentence_support_information", [])
    adh = (sum(1 for s in si if s.get("supported", False)) / len(si)
           if si else (1.0 if j.get("overall_supported", False) else 0.0))
    return {"context_relevance": round(ctx_rel,4),
            "context_utilization": round(ctx_util,4),
            "completeness": round(complt,4),
            "adherence": round(adh,4)}

print("compute_metrics(judge_output) defined")
print("  Relevance   = |relevant| / total_context")
print("  Utilization = |utilized| / total_context")
print("  Completeness = (util AND rel) / relevant")
print("  Adherence   = supported_sents / total_response_sents")


compute_metrics(judge_output) defined
  Relevance   = |relevant| / total_context
  Utilization = |utilized| / total_context
  Completeness = (util AND rel) / relevant
  Adherence   = supported_sents / total_response_sents


---
# 📌 Section 9 — Run Full Evaluation  ✅ *Checkpointed*
> **Option 1**: Results CSV auto-saved to Drive and reloaded next session.
> **Option 2**: Results CSV downloaded to your machine; upload prompt next session.


In [ ]:
# SECTION 9A: Evaluation Function

def evaluate_pipeline(dataset, num_samples, use_reranker=USE_RERANKER):
    """Retrieve -> Generate -> Judge -> Metrics for num_samples."""
    results, errors = [], 0
    print(f"Evaluating {num_samples} samples...")
    print(f"  Reranker: {'ON' if use_reranker else 'OFF'} | LLM: {LLM_MODEL}")
    print(f"  Est. time: {num_samples*5}-{num_samples*12}s\n")

    for i in range(min(num_samples, len(dataset))):
        s = dataset[i]
        q = s["question"]
        print(f"  [{i+1}/{num_samples}] {q[:60]}...")
        try:
            ctx, scores, _ = (retrieve_and_rerank(q) if use_reranker else retrieve(q))
            answer = generate_answer(q, ctx)
            judge  = judge_response(q, ctx, answer)
            pred   = compute_metrics(judge)
            gt_adh = s.get("adherence_score", 0)
            if isinstance(gt_adh, bool): gt_adh = 1.0 if gt_adh else 0.0
            results.append({
                "index": i, "question": q, "answer": answer,
                "top_retrieval_score": scores[0] if scores else 0,
                "pred_relevance": pred["context_relevance"],
                "pred_utilization": pred["context_utilization"],
                "pred_completeness": pred["completeness"],
                "pred_adherence": pred["adherence"],
                "gt_relevance": s.get("relevance_score", 0),
                "gt_utilization": s.get("utilization_score", 0),
                "gt_completeness": s.get("completeness_score", 0),
                "gt_adherence": gt_adh,
            })
            time.sleep(2)
        except Exception as e:
            print(f"  Sample {i} error: {e}")
            errors += 1
            time.sleep(4)

    print(f"\n{len(results)}/{num_samples} succeeded, {errors} errors.")
    return pd.DataFrame(results)

print("evaluate_pipeline() defined")


evaluate_pipeline() defined


In [ ]:
# SECTION 9B: Run or Load Evaluation (checkpoint-guarded)

results_df = load_results(RESULTS_FILE)

if results_df is None and STORAGE_OPTION == 2:
    print(f"No results in session. Upload '{RESULTS_FNAME}' if available.")
    results_df = prompt_upload_results(RESULTS_FILE)

if results_df is not None:
    print(f"Results loaded: {len(results_df)} samples")
else:
    print("Running full evaluation...")
    results_df = evaluate_pipeline(ds, num_samples=NUM_EVAL_SAMPLES)
    save_results(results_df, RESULTS_FILE)

cols = ["pred_relevance","pred_utilization","pred_completeness","pred_adherence",
        "gt_relevance","gt_utilization","gt_completeness","gt_adherence"]
print("\nRESULTS PREVIEW (first 5)")
print(results_df[cols].head().to_string())


Results loaded: 50 samples

RESULTS PREVIEW (first 5)
   pred_relevance  pred_utilization  pred_completeness  pred_adherence  gt_relevance  gt_utilization  gt_completeness  gt_adherence
0          0.0000            0.0000             0.0000             0.0      0.111111        0.111111              1.0           1.0
1          0.1818            0.0909             0.5000             0.4      0.040000        0.040000              1.0           1.0
2          0.2000            0.4000             1.0000             1.0      0.100000        0.050000              0.5           1.0
3          0.0909            0.0909             1.0000             1.0      0.111111        0.111111              1.0           0.0
4          0.2143            0.0714             0.3333             1.0      0.050000        0.050000              1.0           1.0


---
# 📌 Section 10 — RMSE & AUC-ROC Scores
> Final evaluation numbers for your report.


In [ ]:
# SECTION 10: Compute RMSE and AUC-ROC
def compute_evaluation_scores(df, label=""):
    metrics = ["relevance","utilization","completeness","adherence"]
    print("=" * 70)
    print(f"EVALUATION SCORES {label}")
    print("=" * 70)
    print(f"{'Metric':<22} {'RMSE down':>10} {'AUC-ROC up':>12} {'Pred mean':>10} {'GT mean':>8}")
    print("-" * 65)
    rmse_r, auc_r = {}, {}
    for m in metrics:
        pred = df[f"pred_{m}"].values
        gt   = df[f"gt_{m}"].values
        rmse = np.sqrt(mean_squared_error(gt, pred))
        rmse_r[m] = rmse
        gt_bin = (gt >= 0.5).astype(int)
        if len(np.unique(gt_bin)) < 2:
            auc_str, auc_r[m] = "N/A", None
        else:
            try:
                auc = roc_auc_score(gt_bin, pred)
                auc_r[m] = auc
                auc_str = f"{auc:.4f}"
            except Exception:
                auc_str, auc_r[m] = "Error", None
        print(f"{m:<22} {rmse:>10.4f} {auc_str:>12} {np.mean(pred):>10.3f} {np.mean(gt):>8.3f}")
    avg_rmse = np.mean(list(rmse_r.values()))
    valid = [v for v in auc_r.values() if v is not None]
    avg_auc = f"{np.mean(valid):.4f}" if valid else "N/A"
    print("-" * 65)
    print(f"{'AVERAGE':<22} {avg_rmse:>10.4f} {avg_auc:>12}")
    print("=" * 70)
    return rmse_r, auc_r

rmse_results, auc_results = compute_evaluation_scores(
    results_df,
    label=f"[{DATASET_NAME} | {EMBED_MODEL_NAME} | {'reranked' if USE_RERANKER else 'faiss-only'}]"
)


EVALUATION SCORES [finqa | all-MiniLM-L6-v2 | reranked]
Metric                  RMSE down   AUC-ROC up  Pred mean  GT mean
-----------------------------------------------------------------
relevance                  0.2132       0.2031      0.158    0.091
utilization                0.1607          N/A      0.139    0.061
completeness               0.4796       0.3778      0.823    0.864
adherence                  0.5265       0.3778      0.808    0.900
-----------------------------------------------------------------
AVERAGE                    0.3450       0.3196


---
# 📌 Section 11 — Multi-Dataset Comparison
> Run after results exist for at least 2 datasets.
> **Option 1**: Loads CSVs from Drive automatically.
> **Option 2**: Prompts upload for each dataset's results CSV.


In [ ]:
# SECTION 11: Cross-Dataset Comparison Table

DATASETS_TO_COMPARE = ["finqa", "hotpotqa", "emanual"]  # update as needed
comparison_rows = []

for ds_name in DATASETS_TO_COMPARE:
    csv_name = f"results_{EMBED_SAFE}_top{TOP_K_FINAL}_n{NUM_EVAL_SAMPLES}.csv"
    local_path = f"/content/rag_cache/{ds_name}/{csv_name}"
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    if STORAGE_OPTION == 1:
        drive_path = f"/content/drive/MyDrive/{DRIVE_PROJECT_FOLDER}/{ds_name}/{csv_name}"
        if not os.path.exists(local_path) and os.path.exists(drive_path):
            shutil.copy2(drive_path, local_path)
    elif STORAGE_OPTION == 2 and not os.path.exists(local_path):
        print(f"Upload results CSV for '{ds_name}' ({csv_name}):")
        from google.colab import files as colab_files
        uploaded = colab_files.upload()
        if csv_name in uploaded:
            with open(local_path, "wb") as f:
                f.write(uploaded[csv_name])

    if not os.path.exists(local_path):
        print(f"  No results for '{ds_name}'. Skipping.")
        continue

    df = pd.read_csv(local_path)
    row = {"dataset": ds_name, "n_samples": len(df)}
    for m in ["relevance","utilization","completeness","adherence"]:
        pred = df[f"pred_{m}"].values
        gt   = df[f"gt_{m}"].values
        row[f"{m}_rmse"]      = round(np.sqrt(mean_squared_error(gt, pred)), 4)
        row[f"{m}_pred_mean"] = round(np.mean(pred), 3)
        row[f"{m}_gt_mean"]   = round(np.mean(gt), 3)
    comparison_rows.append(row)

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows)
    print("=" * 80)
    print("CROSS-DATASET COMPARISON  (lower RMSE = better alignment with ground truth)")
    print("=" * 80)
    print(comp_df.to_string(index=False))
else:
    print("No multi-dataset results found yet.")
    print("Run notebook with different DATASET_NAME values in Section 0 first.")


---
# 📌 Section 12 — Gradio Demo
> Interactive web app. Run last.


In [ ]:
# SECTION 12: Gradio Interactive Demo
import gradio as gr

def rag_ui(question, use_rr, top_k_ui):
    top_k_ui = int(top_k_ui)
    ctx, scores, _ = (retrieve_and_rerank(question, final_top_k=top_k_ui)
                      if use_rr and reranker is not None
                      else retrieve(question, top_k=top_k_ui))
    method = "FAISS + Reranker" if (use_rr and reranker) else "FAISS Only"
    answer = generate_answer(question, ctx)
    ctx_md = ""
    for i, (c, s) in enumerate(zip(ctx, scores), 1):
        ctx_md += f"**Chunk {i}** (score: {s:.4f})\n{c[:400]}{'...' if len(c)>400 else ''}\n\n---\n\n"
    meta = (f"**Dataset:** {DATASET_NAME}  |  **Method:** {method}\n"
            f"**Embed:** {EMBED_MODEL_NAME}  |  **LLM:** {LLM_MODEL}\n"
            f"**Top score:** {scores[0]:.4f}  |  "
            f"**Storage:** Option {STORAGE_OPTION}")
    return answer, ctx_md, meta

with gr.Blocks(title=f"RAG - {DATASET_NAME}") as demo:
    gr.Markdown(
        f"# RAG System - {DATASET_NAME.upper()}\n"
        f"**Dataset:** RAGBench {DATASET_NAME} ({len(ds)} samples) | "
        f"**Chunks:** {len(chunks)} | **Embed:** {EMBED_MODEL_NAME} | **LLM:** {LLM_MODEL}"
    )
    with gr.Row():
        with gr.Column(scale=2):
            q_box = gr.Textbox(label="Question", placeholder="Ask anything...", lines=2)
            with gr.Row():
                rr_cb = gr.Checkbox(label="Use Reranker", value=USE_RERANKER)
                tk_sl = gr.Slider(3, 10, value=TOP_K_FINAL, step=1, label="Chunks")
            btn = gr.Button("Get Answer", variant="primary")
        with gr.Column(scale=1):
            meta_out = gr.Markdown()
    ans_out = gr.Textbox(label="RAG Answer", lines=4, interactive=False)
    with gr.Accordion("Retrieved Context", open=False):
        ctx_out = gr.Markdown()
    gr.Markdown("### Sample Questions")
    gr.Examples([[ds[i]["question"]] for i in range(min(5, len(ds)))], inputs=q_box)
    btn.click(fn=rag_ui, inputs=[q_box, rr_cb, tk_sl], outputs=[ans_out, ctx_out, meta_out])

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a260ed1b09cf515335.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a260ed1b09cf515335.gradio.live
